# 03 — DistilBERT Fine-Tuning for SMS Spam Detection

## 1. Project Objective

The objective of this notebook is to fine-tune a pretrained DistilBERT transformer model to classify SMS messages as either legitimate (ham) or spam.

This experiment builds on the initial exploratory data analysis and the TF-IDF + Logistic Regression baseline developed earlier in the project.Untitled

## 2. Dataset

- **Source:** UCI SMS Spam Collection
- **Task:** Binary text classification
- **Input:** SMS message text
- **Target:** Message label (`ham` or `spam`)
- **Class mapping:** `ham = 0`, `spam = 1`
- **Dataset size after cleaning:** 5,169 messages

## 3. Experimental Workflow

1. Import required libraries and verify the environment.
2. Load the original dataset and reproduce the established cleaning procedure.
3. Split the dataset into training, validation, and test sets using stratified sampling.
4. Encode the target labels as numeric class IDs.
5. Load the pretrained DistilBERT tokenizer.
6. Tokenize the SMS messages and prepare input tensors.
7. Load the pretrained DistilBERT model with a sequence-classification head.
8. Fine-tune the model using the training set.
9. Monitor validation performance and select the best model checkpoint.
10. Evaluate the selected model on the held-out test set.
11. Compare the results with the TF-IDF + Logistic Regression baseline.
12. Save the trained model and tokenizer for later application integration.

## 4. Evaluation Metrics

The model will be evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

Particular attention will be given to spam recall and spam precision because both missed spam messages and legitimate messages incorrectly flagged as spam matter in a practical spam detector.

## 5. Reproducibility and Data Integrity

- Use a fixed random seed for reproducible data splitting and training where applicable.
- Use stratified sampling to preserve class proportions.
- Keep training, validation, and test data separate.
- Use the validation set for model-development decisions.
- Reserve the test set for final evaluation.
- Keep raw and processed datasets out of version control when excluded by `.gitignore`.
- Record model configuration, training settings, and evaluation results.

## 6. Expected Outcome

A fine-tuned DistilBERT spam classifier, a documented evaluation against the classical machine-learning baseline, and saved model artifacts that can later be integrated into a FastAPI application.


In [1]:
# Python standard library
import random
import json
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Deep learning
import torch
from torch.utils.data import Dataset, DataLoader

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)


In [2]:
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\nAll packages imported successfully!")

PyTorch: 2.15.0+cpu
NumPy: 2.5.2
Pandas: 3.0.5
CUDA build: None
CUDA available: False

All packages imported successfully!


In [3]:
# Locate  raw dataset
data_path = Path("../data/raw/SMSSpamCollection")

# whether the file exists
print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path.resolve())

Dataset exists: True
Dataset path: /home/student/email-spam-detector/data/raw/SMSSpamCollection


In [4]:
df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

Dataset shape: (5572, 2)

First five rows:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
print("Dataset information:")
df.info()

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64


In [6]:
# Convert text labels into numerical labels
label_mapping = {
    "ham": 0,
    "spam": 1
}

#creates a new column containing those numerical labels
df["label_id"] = df["label"].map(label_mapping)

# Inspect the conversion
print(df[["label", "label_id"]].head())

# Check for labels that were not mapped
print("\nMissing numerical labels:", df["label_id"].isnull().sum())

  label  label_id
0   ham         0
1   ham         0
2  spam         1
3   ham         0
4   ham         0

Missing numerical labels: 0


In [7]:
X = df["message"]
y = df["label_id"]

print("Messages:", X.shape)
print("Labels:", y.shape)

Messages: (5572,)
Labels: (5572,)


In [8]:
# First split: 70% training, 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Total samples:", len(X_train) + len(X_val) + len(X_test))

Training samples: 3900
Validation samples: 836
Testing samples: 836
Total samples: 5572


In [9]:
# Check class proportions in each dataset split
print("Training distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training distribution:
label_id
0    0.865897
1    0.134103
Name: proportion, dtype: float64

Validation distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64

Testing distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64


In [10]:
# Load the pretrained DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522


In [15]:
# Tokenize training messages
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize validation messages
val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    max_length=128
)

# Tokenize test messages
test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    max_length=128
)

In [16]:
print("Training messages:", len(train_encodings["input_ids"]))
print("Validation messages:", len(val_encodings["input_ids"]))
print("Test messages:", len(test_encodings["input_ids"]))

print("\nFirst training message token IDs:")
print(train_encodings["input_ids"][0])

print("\nFirst training message attention mask:")
print(train_encodings["attention_mask"][0])

Training messages: 3900
Validation messages: 836
Test messages: 836

First training message token IDs:
[101, 3125, 999, 9433, 1018, 1006, 2888, 1010, 1021, 1058, 6220, 1016, 2888, 7644, 2007, 1037, 3722, 2915, 2013, 1020, 4210, 2013, 1037, 3413, 2011, 15214, 27052, 2361, 2000, 2507, 9433, 1037, 1016, 3125, 7785, 2044, 6275, 8117, 2015, 1012, 102]

First training message attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [17]:
class SMSDataset(Dataset):

    def __init__(self, encodings, labels):
        # Store the tokenized encodings in this object
        self.encodings = encodings

        # Convert labels to a Python list and store them
        self.labels = labels.tolist()

    def __len__(self):
        # Return the number of labels stored in this object
        return len(self.labels)

    def __getitem__(self, idx):
        # Retrieve the input IDs for the message at index idx
        input_ids = self.encodings["input_ids"][idx]

        # Retrieve the attention mask for the message at index idx
        attention_mask = self.encodings["attention_mask"][idx]

        # Retrieve the label for the message at index idx
        label = self.labels[idx]

        # Convert input IDs into a PyTorch tensor
        input_ids = torch.tensor(input_ids, dtype=torch.long)

        # Convert attention mask into a PyTorch tensor
        attention_mask = torch.tensor(attention_mask, dtype=torch.long)

        # Convert the label into a PyTorch tensor
        label = torch.tensor(label, dtype=torch.long)

        # Return a dictionary containing the tensors
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label
        }

In [18]:
train_dataset = SMSDataset(train_encodings, y_train)

val_dataset = SMSDataset(val_encodings, y_val)

test_dataset = SMSDataset(test_encodings, y_test)

print(
    "Training length:", len(train_dataset),
    "\nValidation length:", len(val_dataset),
    "\nTesting length:", len(test_dataset)
)

Training length: 3900 
Validation length: 836 
Testing length: 836


In [19]:
sample = train_dataset[0]

print(sample.items())
print(sample.values())
print(sample.keys())

dict_items([('input_ids', tensor([  101,  3125,   999,  9433,  1018,  1006,  2888,  1010,  1021,  1058,
         6220,  1016,  2888,  7644,  2007,  1037,  3722,  2915,  2013,  1020,
         4210,  2013,  1037,  3413,  2011, 15214, 27052,  2361,  2000,  2507,
         9433,  1037,  1016,  3125,  7785,  2044,  6275,  8117,  2015,  1012,
          102])), ('attention_mask', tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])), ('labels', tensor(1))])
dict_values([tensor([  101,  3125,   999,  9433,  1018,  1006,  2888,  1010,  1021,  1058,
         6220,  1016,  2888,  7644,  2007,  1037,  3722,  2915,  2013,  1020,
         4210,  2013,  1037,  3413,  2011, 15214, 27052,  2361,  2000,  2507,
         9433,  1037,  1016,  3125,  7785,  2044,  6275,  8117,  2015,  1012,
          102]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [20]:
print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label shape:", sample["labels"].shape)

Input IDs shape: torch.Size([41])
Attention mask shape: torch.Size([41])
Label shape: torch.Size([])


In [21]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [22]:
data_collator 

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [23]:
BATCH_SIZE = 16

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [24]:
batch = next(iter(train_dataloader))

print("\nIndividual values")
print(batch["input_ids"][:2])
print(batch["attention_mask"][:2])
print(batch["labels"][:2])
print("\nShapes")
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)


Individual values
tensor([[  101,  2123,  2102,  2228,  2017,  2342,  3756,  4003,  2005,  2866,
          3604,  1012,  3198,  2619,  2008,  2038,  2908,  2077,  1012,  2065,
          2017,  2079,  2049,  2074,  1004,  8318,  1025,  1001,  1004, 14181,
          1025, 14189,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,    

In [25]:
print("Real tokens in message 1:", batch["attention_mask"][0].sum().item())
print("Real tokens in message 2:", batch["attention_mask"][1].sum().item())

Real tokens in message 1: 33
Real tokens in message 2: 16


In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [28]:
print("Total parameters:", sum(p.numel() for p in model.parameters()))

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

Total parameters: 66955010
Trainable parameters: 66955010


In [29]:
model.train()

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

print(outputs.logits)
print(outputs.logits.shape)

tensor([[-0.0170, -0.1122],
        [-0.0854, -0.0056],
        [-0.1782, -0.0779],
        [-0.0168, -0.0540],
        [ 0.0516, -0.1900],
        [-0.0082, -0.0105],
        [-0.0127, -0.0921],
        [-0.1287, -0.1181],
        [-0.0844, -0.1457],
        [-0.1329, -0.1231],
        [ 0.0473, -0.0843],
        [-0.0767, -0.1320],
        [-0.1636, -0.0950],
        [-0.0987, -0.0821],
        [ 0.0910, -0.2605],
        [ 0.0157, -0.1041]], grad_fn=<AddmmBackward0>)
torch.Size([16, 2])


In [30]:
print(batch["labels"])

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0])


In [31]:
loss_function = torch.nn.CrossEntropyLoss()

In [32]:
loss = loss_function(outputs.logits, batch["labels"])

print("Loss:", loss.item())

Loss: 0.6887252330780029


In [33]:
### The computational sequence is:
# optimizer.zero_grad()  # Clear previous gradients

# outputs = model(...)   # Forward propagation

# loss = loss_function(...)  # Calculate loss

# loss.backward()        # Calculate gradients

# optimizer.step()       # Update model parameters

In [34]:
print(model.classifier.weight.grad) #Checking the gradient before backpropagation

None


In [35]:
loss.backward()


In [36]:
print(model.classifier.weight.grad.shape)

torch.Size([2, 768])


In [37]:
print(model.classifier.weight.grad)  # Inspecting the actual gradient values

tensor([[-0.1231, -0.1860, -0.0011,  ..., -0.0226, -0.0185,  0.0000],
        [ 0.1231,  0.1860,  0.0011,  ...,  0.0226,  0.0185,  0.0000]])


In [38]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=1e-2
)

In [39]:
optimizer.zero_grad() #clear gradient

In [40]:
print(model.classifier.weight.grad)

None


In [41]:
EPOCHS = 3   #3 means the model will process all 3,900 training messages three times

In [42]:
def train_one_epoch(model, dataloader, optimizer, loss_function):

    model.train()
    model.zero_grad(set_to_none=True)
    total_loss = 0

    for batch_number, batch in enumerate(dataloader, start=1):

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        loss = loss_function(outputs.logits, batch["labels"])

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch["labels"].size(0)
        
        if batch_number % 20 == 0: # checks the remainder when the batch number is divided by 20
            print(
                f"Processed {batch_number}/{len(dataloader)} batches | "
                f"Batch Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(dataloader.dataset)   #divides the accumulated loss by the number of examples

    return average_loss

In [43]:
training_loss = train_one_epoch(
    model,
    train_dataloader,
    optimizer,
    loss_function
)

print(f"\nAverage Training Loss: {training_loss:.4f}")

Processed 20/244 batches | Batch Loss: 0.2690
Processed 40/244 batches | Batch Loss: 0.0581
Processed 60/244 batches | Batch Loss: 0.0448
Processed 80/244 batches | Batch Loss: 0.0495
Processed 100/244 batches | Batch Loss: 0.0096
Processed 120/244 batches | Batch Loss: 0.0381
Processed 140/244 batches | Batch Loss: 0.0132
Processed 160/244 batches | Batch Loss: 0.0516
Processed 180/244 batches | Batch Loss: 0.0090
Processed 200/244 batches | Batch Loss: 0.0040
Processed 220/244 batches | Batch Loss: 0.0032
Processed 240/244 batches | Batch Loss: 0.0045

Average Training Loss: 0.0879


In [44]:
def evaluate_model(model, dataloader, loss_function):
    model.eval()
    
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            loss = loss_function(outputs.logits, batch["labels"])

            total_loss += loss.item() * batch["labels"].size(0)

            batch_predictions = torch.argmax(outputs.logits, dim=1)
            
            predictions.extend(batch_predictions.cpu().tolist())
            true_labels.extend(batch["labels"].cpu().tolist())
            
    average_loss = total_loss / len(dataloader.dataset)
    
    return average_loss, predictions, true_labels

In [46]:
validation_loss, predictions, true_labels = evaluate_model(
    model,
    val_dataloader,
    loss_function               #python unpacking
)

print(f"Validation Loss: {validation_loss:.4f}")


Validation Loss: 0.0268


In [47]:
accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(true_labels, predictions, zero_division=0)

recall = recall_score(true_labels, predictions, zero_division=0)

f1 = f1_score(true_labels, predictions, zero_division=0)

In [48]:
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.9916
Precision: 0.9565
Recall:    0.9821
F1-score:  0.9692


In [49]:
training_loss = train_one_epoch(
    model,
    train_dataloader,
    optimizer,
    loss_function
)

print(f"\nEpoch 2 Training Loss: {training_loss:.4f}")

Processed 20/244 batches | Batch Loss: 0.0053
Processed 40/244 batches | Batch Loss: 0.0042
Processed 60/244 batches | Batch Loss: 0.0025
Processed 80/244 batches | Batch Loss: 0.0046
Processed 100/244 batches | Batch Loss: 0.0022
Processed 120/244 batches | Batch Loss: 0.0028
Processed 140/244 batches | Batch Loss: 0.0018
Processed 160/244 batches | Batch Loss: 0.0018
Processed 180/244 batches | Batch Loss: 0.0036
Processed 200/244 batches | Batch Loss: 0.0075
Processed 220/244 batches | Batch Loss: 0.0075
Processed 240/244 batches | Batch Loss: 0.0579

Epoch 2 Training Loss: 0.0202


In [66]:
validation_loss, predictions, true_labels = evaluate_model(
    model,
    val_dataloader,
    loss_function
)
print(f"Epoch 2 Validation Loss: {validation_loss:.4f}")

Epoch 2 Validation Loss: 0.0390


In [51]:
accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(true_labels, predictions, zero_division=0)

recall = recall_score(true_labels, predictions, zero_division=0)

f1 = f1_score(true_labels, predictions, zero_division=0)

In [52]:
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.9916
Precision: 1.0000
Recall:    0.9375
F1-score:  0.9677


In [57]:
cm = confusion_matrix(true_labels, predictions)
print(cm)

[[724   0]
 [  7 105]]


In [58]:
training_loss = train_one_epoch(
    model,
    train_dataloader,
    optimizer,
    loss_function
)

print(f"\nEpoch 3 Training Loss: {training_loss:.4f}")

Processed 20/244 batches | Batch Loss: 0.0007
Processed 40/244 batches | Batch Loss: 0.0013
Processed 60/244 batches | Batch Loss: 0.0012
Processed 80/244 batches | Batch Loss: 0.0006
Processed 100/244 batches | Batch Loss: 0.0006
Processed 120/244 batches | Batch Loss: 0.0007
Processed 140/244 batches | Batch Loss: 0.0005
Processed 160/244 batches | Batch Loss: 0.0008
Processed 180/244 batches | Batch Loss: 0.0013
Processed 200/244 batches | Batch Loss: 0.1641
Processed 220/244 batches | Batch Loss: 0.0832
Processed 240/244 batches | Batch Loss: 0.0012

Epoch 3 Training Loss: 0.0074


In [65]:
validation_loss, predictions, true_labels = evaluate_model(
    model,
    val_dataloader,
    loss_function
)
print(f"Epoch 3 Validation Loss: {validation_loss:.4f}")

Epoch 3 Validation Loss: 0.0390


In [62]:
accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(true_labels, predictions, zero_division=0)

recall = recall_score(true_labels, predictions, zero_division=0)

f1 = f1_score(true_labels, predictions, zero_division=0)

In [63]:
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.9904
Precision: 0.9906
Recall:    0.9375
F1-score:  0.9633


In [64]:
cm = confusion_matrix(true_labels, predictions)
print(cm)

[[723   1]
 [  7 105]]


In [67]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [68]:
set_seed()

In [69]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [70]:
print(model.config.num_labels)

2


In [72]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [74]:
best_model_dir = Path("models/best_distilbert")

In [75]:
best_f1 = -1.0

In [78]:
NUM_EPOCHS = 3
history = []

for epoch in range(1, NUM_EPOCHS + 1):

    print(f"\n{'=' * 50}")
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(f"{'=' * 50}")
    
    training_loss = train_one_epoch(
        model,
        train_dataloader,
        optimizer,
        loss_function
    )
    
    validation_loss, predictions, true_labels = evaluate_model(
        model,
        val_dataloader,
        loss_function
    )
    
    
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, zero_division=0)
    recall = recall_score(true_labels, predictions, zero_division=0)
    f1 = f1_score(true_labels, predictions, zero_division=0)

    cm = confusion_matrix(true_labels, predictions)
    
    epoch_results = {
        "epoch": epoch,
        "training_loss": training_loss,
        "validation_loss": validation_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm.tolist()
    }

    history.append(epoch_results)

    print(f"Training Loss:   {training_loss:.4f}")
    print(f"Validation Loss: {validation_loss:.4f}")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Precision:       {precision:.4f}")
    print(f"Recall:          {recall:.4f}")
    print(f"F1 Score:        {f1:.4f}")

    print("\nConfusion Matrix:")
    print(cm)
    
    if f1 > best_f1:
        best_f1 = f1

        model.save_pretrained(str(best_model_dir))
        tokenizer.save_pretrained(str(best_model_dir))

        print(f"\nNew best model saved! F1: {best_f1:.4f}")


Epoch 1/3
Processed 20/244 batches | Batch Loss: 0.3425
Processed 40/244 batches | Batch Loss: 0.0441
Processed 60/244 batches | Batch Loss: 0.0233
Processed 80/244 batches | Batch Loss: 0.0083
Processed 100/244 batches | Batch Loss: 0.1805
Processed 120/244 batches | Batch Loss: 0.0083
Processed 140/244 batches | Batch Loss: 0.0134
Processed 160/244 batches | Batch Loss: 0.0747
Processed 180/244 batches | Batch Loss: 0.3112
Processed 200/244 batches | Batch Loss: 0.0079
Processed 220/244 batches | Batch Loss: 0.0072
Processed 240/244 batches | Batch Loss: 0.0087
Training Loss:   0.0921
Validation Loss: 0.0396
Accuracy:        0.9880
Precision:       0.9904
Recall:          0.9196
F1 Score:        0.9537

Confusion Matrix:
[[723   1]
 [  9 103]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


New best model saved! F1: 0.9537

Epoch 2/3
Processed 20/244 batches | Batch Loss: 0.0044
Processed 40/244 batches | Batch Loss: 0.0054
Processed 60/244 batches | Batch Loss: 0.0068
Processed 80/244 batches | Batch Loss: 0.2054
Processed 100/244 batches | Batch Loss: 0.0047
Processed 120/244 batches | Batch Loss: 0.0044
Processed 140/244 batches | Batch Loss: 0.0210
Processed 160/244 batches | Batch Loss: 0.2132
Processed 180/244 batches | Batch Loss: 0.0071
Processed 200/244 batches | Batch Loss: 0.0046
Processed 220/244 batches | Batch Loss: 0.0014
Processed 240/244 batches | Batch Loss: 0.0033
Training Loss:   0.0237
Validation Loss: 0.0322
Accuracy:        0.9928
Precision:       0.9649
Recall:          0.9821
F1 Score:        0.9735

Confusion Matrix:
[[720   4]
 [  2 110]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


New best model saved! F1: 0.9735

Epoch 3/3
Processed 20/244 batches | Batch Loss: 0.0026
Processed 40/244 batches | Batch Loss: 0.0087
Processed 60/244 batches | Batch Loss: 0.0014
Processed 80/244 batches | Batch Loss: 0.0029
Processed 100/244 batches | Batch Loss: 0.0011
Processed 120/244 batches | Batch Loss: 0.0026
Processed 140/244 batches | Batch Loss: 0.0011
Processed 160/244 batches | Batch Loss: 0.0034
Processed 180/244 batches | Batch Loss: 0.0019
Processed 200/244 batches | Batch Loss: 0.0047
Processed 220/244 batches | Batch Loss: 0.0025
Processed 240/244 batches | Batch Loss: 0.0011
Training Loss:   0.0119
Validation Loss: 0.0388
Accuracy:        0.9904
Precision:       1.0000
Recall:          0.9286
F1 Score:        0.9630

Confusion Matrix:
[[724   0]
 [  8 104]]


In [79]:
for result in history:
    print(result)

{'epoch': 1, 'training_loss': 0.09213975780404722, 'validation_loss': 0.03957474463436592, 'accuracy': 0.9880382775119617, 'precision': 0.9903846153846154, 'recall': 0.9196428571428571, 'f1': 0.9537037037037037, 'confusion_matrix': [[723, 1], [9, 103]]}
{'epoch': 2, 'training_loss': 0.02374458985737501, 'validation_loss': 0.032222581737670314, 'accuracy': 0.992822966507177, 'precision': 0.9649122807017544, 'recall': 0.9821428571428571, 'f1': 0.9734513274336283, 'confusion_matrix': [[720, 4], [2, 110]]}
{'epoch': 3, 'training_loss': 0.011877234141605023, 'validation_loss': 0.03883279084019657, 'accuracy': 0.9904306220095693, 'precision': 1.0, 'recall': 0.9285714285714286, 'f1': 0.9629629629629629, 'confusion_matrix': [[724, 0], [8, 104]]}


In [80]:
history_path = Path("models/training_history.json")

with open(history_path, "w") as file:
    json.dump(history, file, indent=4)

In [81]:
print(history_path.exists())

True


In [82]:
for path in Path("models").rglob("*"):
    print(path)

models/best_distilbert
models/training_history.json
models/best_distilbert/tokenizer_config.json
models/best_distilbert/model.safetensors
models/best_distilbert/tokenizer.json
models/best_distilbert/config.json


In [83]:
best_model = AutoModelForSequenceClassification.from_pretrained(
    str(best_model_dir)
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Preparing model for testing

In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_model.to(device)
best_model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [85]:
def evaluate_model(model, dataloader, loss_function, device):

    model.eval()

    total_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_function(outputs.logits, labels)

            total_loss += loss.item() * labels.size(0)

            batch_predictions = torch.argmax(outputs.logits, dim=1)

            predictions.extend(batch_predictions.cpu().tolist())
            true_labels.extend(labels.cpu().tolist())

    average_loss = total_loss / len(dataloader.dataset)

    return average_loss, predictions, true_labels

In [86]:
test_loss, test_predictions, test_true_labels = evaluate_model(
    best_model,
    test_dataloader,
    loss_function,
    device
)

In [87]:
test_accuracy = accuracy_score(test_true_labels, test_predictions)

test_precision = precision_score(
    test_true_labels, test_predictions, zero_division=0
)

test_recall = recall_score(
    test_true_labels, test_predictions, zero_division=0
)

test_f1 = f1_score(
    test_true_labels, test_predictions, zero_division=0
)

test_cm = confusion_matrix(test_true_labels, test_predictions)

In [88]:
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")

print("\nTest Confusion Matrix:")
print(test_cm)

print("\nClassification Report:")
print(
    classification_report(
        test_true_labels,
        test_predictions,
        target_names=["HAM", "SPAM"],
        zero_division=0
    )
)

Test Loss:      0.0403
Test Accuracy:  0.9880
Test Precision: 0.9397
Test Recall:    0.9732
Test F1 Score:  0.9561

Test Confusion Matrix:
[[717   7]
 [  3 109]]

Classification Report:
              precision    recall  f1-score   support

         HAM       1.00      0.99      0.99       724
        SPAM       0.94      0.97      0.96       112

    accuracy                           0.99       836
   macro avg       0.97      0.98      0.97       836
weighted avg       0.99      0.99      0.99       836



## Validation Error Analysis

### Objective
Examine the validation examples that DistilBERT misclassified to understand the model's weaknesses and identify possible areas for improvement.

### Why Validation Data?
We use the validation set for error analysis because it allows us to investigate model errors and guide future improvements without compromising the independence of our final test evaluation.

### Analysis Plan
1. Generate predictions on the validation dataset.
2. Compare predicted labels with actual labels.
3. Identify incorrectly classified SMS messages.
4. Examine the messages to understand possible causes of misclassification.
5. Document findings and potential improvements.

### Important Note
The test set remains untouched during model development. It will only be used for final evaluation after all model decisions have been completed.

**Label Mapping:**
- `0` = HAM
- `1` = SPAM


In [90]:
val_loss, val_predictions, val_true_labels = evaluate_model(
    best_model,
    val_dataloader,
    loss_function,
    device
)

In [91]:
error_analysis = pd.DataFrame({
    "message": X_val.tolist(),
    "actual": y_val.tolist(),
    "predicted": val_predictions
})

In [92]:
misclassified = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
]

print(f"Misclassified messages: {len(misclassified)}")

Misclassified messages: 6


In [93]:
print(misclassified.to_string(index=False))

                                                                                                                                                 message  actual  predicted
             We are pleased to inform that your application for Airtel Broadband is processed successfully. Your installation will happen within 3 days.       0          1
                                                                                      they released another Italian one today and it has a cosign option       0          1
                                           Also north carolina and texas atm, you would just go to the gre site and pay for the test results to be sent.       0          1
                                                                        Somebody set up a website where you can play hold em using eve online spacebucks       0          1
                                                                                                                RCT' THNQ Adrian for U text.

## Validation Error Analysis — Findings

The DistilBERT model misclassified 6 out of 836 validation examples.

### False Positives (4)
Four legitimate HAM messages were incorrectly classified as SPAM. These messages included commercial notifications, payment references, website mentions, and unfamiliar conversational contexts.

### False Negatives (2)
Two SPAM messages were incorrectly classified as HAM. One used abbreviated SMS language, while the other used conversational and flattering language.

### Key Observations
1. Commercial vocabulary does not necessarily indicate spam.
2. Legitimate messages can contain payment references and website mentions.
3. Spam messages do not always contain obvious promotional or financial keywords.
4. Context and language patterns remain important challenges for text classification.

### Conclusion
The model achieved strong validation performance, but these errors demonstrate that high classification accuracy does not imply perfect classification. Further improvements should be evaluated using validation data without compromising the independent test set.